In [3]:
# Imports and setup
import nest_asyncio
nest_asyncio.apply()

import tensorflow as tf
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import socketio
import asyncio
import threading
from datetime import datetime

# Verify GPU
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# Server setup
sio = socketio.AsyncServer(async_mode='asgi', cors_allowed_origins='*')
app = FastAPI()
app.mount("/socket.io", socketio.ASGIApp(sio))

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global training control
training_active = False
current_model = None
# Model and dataset setup
def load_hardcoded_dataset():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    x_train = x_train.astype('float32') / 255
    x_test = x_test.astype('float32') / 255
    return (x_train, y_train), (x_test, y_test)

def create_model(model_type='simple_cnn'):
    if model_type == 'simple_cnn':
        model = tf.keras.Sequential([
            tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
            tf.keras.layers.MaxPooling2D((2,2)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(10, activation='softmax')
        ])
    elif model_type == 'resnet':
        base = tf.keras.applications.ResNet50(weights=None, include_top=False, input_shape=(32,32,3))
        model = tf.keras.Sequential([
            base,
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(10, activation='softmax')
        ])
    model.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
    return model
# Training system with real-time updates
class TrainingCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        asyncio.run(sio.emit('training_update', {
            'epoch': epoch + 1,
            'loss': logs['loss'],
            'accuracy': logs['accuracy'],
            'val_loss': logs.get('val_loss', 0),
            'val_accuracy': logs.get('val_accuracy', 0)
        }))

def training_thread(model_type, epochs):
    global training_active, current_model
    try:
        (x_train, y_train), (x_test, y_test) = load_hardcoded_dataset()
        model = create_model(model_type)
        current_model = model
        
        history = model.fit(
            x_train, y_train,
            epochs=epochs,
            validation_data=(x_test, y_test),
            batch_size=256,
            callbacks=[TrainingCallback()]
        )
    finally:
        training_active = False

# Socket.IO handlers
@sio.on('start_training')
async def start_training(sid, data):
    global training_active
    if not training_active:
        training_active = True
        threading.Thread(target=training_thread, 
                        args=(data['model_type'], data['epochs'])).start()
        return {'status': 'started'}
    return {'status': 'already_running'}

@sio.on('predict')
async def handle_predict(sid, data):
    if current_model is None:
        return {'error': 'No model trained yet'}
    img = np.array(data['image']).reshape(1,32,32,3)
    prediction = current_model.predict(img).tolist()[0]
    return {'prediction': prediction}

# Start server
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
    

INFO:     Started server process [14744]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:39984 - "GET /socket.io/?EIO=4&transport=polling&t=PIxgsZk HTTP/1.1" 200 OK
INFO:     127.0.0.1:57060 - "GET /socket.io/?EIO=4&transport=polling&t=PIxgu1U HTTP/1.1" 200 OK
INFO:     127.0.0.1:35560 - "GET /socket.io/?EIO=4&transport=polling&t=PIxgvVE HTTP/1.1" 200 OK
INFO:     127.0.0.1:35568 - "GET /socket.io/?EIO=4&transport=polling&t=PIxgwy- HTTP/1.1" 200 OK
INFO:     127.0.0.1:54042 - "GET /socket.io/?EIO=4&transport=polling&t=PIxgyQk HTTP/1.1" 200 OK
INFO:     127.0.0.1:54044 - "GET /socket.io/?EIO=4&transport=polling&t=PIxgzuU HTTP/1.1" 200 OK
INFO:     127.0.0.1:40668 - "GET /socket.io/?EIO=4&transport=polling&t=PIxg_ME HTTP/1.1" 200 OK
INFO:     127.0.0.1:58936 - "GET /socket.io/?EIO=4&transport=polling&t=PIxh3dH HTTP/1.1" 200 OK
INFO:     127.0.0.1:58936 - "GET /socket.io/?EIO=4&transport=polling&t=PIxh3uP HTTP/1.1" 200 OK
INFO:     127.0.0.1:58936 - "GET /socket.io/?EIO=4&transport=polling&t=PIxh4K3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:58936 - "GET /socket

RuntimeError: asyncio.run() cannot be called from a running event loop